# 03 · Análisis exploratorio (EDA)

Cada prueba responde a una decisión de modelado: tendencia (¿hace falta
diferenciar?), estacionalidad diaria y semanal (¿qué rezagos?), calendario
(¿sirven las banderas de festivo?) y temperatura (¿entra lineal o con la hora?).

Las figuras para el informe las produce `python scripts/eda.py` con las mismas
funciones (`src/mineria/eda.py`).

In [ ]:
import sys, warnings
from pathlib import Path

# Sube desde la carpeta del notebook hasta la raiz del proyecto, la que contiene
# src/ingesta: funciona igual en notebooks/ que en notebooks/limpieza/<dataset>/.
RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "ingesta").is_dir())
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
warnings.filterwarnings("ignore", category=FutureWarning)

DATASETS = RAIZ / "datasets"
print("raiz     :", RAIZ)
print("datasets :", DATASETS, "->", "existe" if DATASETS.exists() else "FALTA")

In [ ]:
PROCESADO = RAIZ / "data" / "procesado"
panel = pd.read_parquet(PROCESADO / "panel_modelado.parquet")
print(f"{len(panel):,} filas x {panel.shape[1]} columnas · {panel.fecha_hora.min()} .. {panel.fecha_hora.max()}")

In [ ]:
from mineria import eda
serie = eda.recortar_parcial(eda.serie_mwh(panel), dias=5)   # publicacion parcial: notas/hallazgos_calidad.md
pd.Series(eda.descriptivos(serie)).to_frame("valor")

## Tendencia y estacionariedad

In [ ]:
print(eda.crecimiento_anual(serie).round(2).to_string())
pd.Series(eda.estacionariedad(serie))

In [ ]:
from statsmodels.tsa.seasonal import STL
diario = serie.resample("D").sum(min_count=24).dropna() / 1000
STL(diario, period=7, robust=True).fit().plot();

## Estacionalidad diaria y semanal

In [ ]:
tabla = serie.groupby([serie.index.dayofweek, serie.index.hour]).median().unstack() / 1000
fig, eje = plt.subplots(figsize=(12, 3.5))
im = eje.imshow(tabla.values, aspect="auto", cmap="Blues")
eje.set_yticks(range(7), ["lun", "mar", "mie", "jue", "vie", "sab", "dom"]); eje.set_xlabel("hora")
fig.colorbar(im, ax=eje, label="GWh"); eje.set_title("Mediana horaria por día y hora");

In [ ]:
acf, pacf = eda.autocorrelacion(serie)
print("ACF  1h  :", round(acf[1], 3))
print("ACF 24h  :", round(acf[24], 3))
print("ACF 168h :", round(acf[168], 3), " <- la misma hora de la semana anterior pesa más que la del día anterior")
acf.plot(figsize=(12, 3), title="ACF hasta 2 semanas");

## Calendario colombiano (Ley Emiliani)

In [ ]:
local = panel.set_index(eda.serie_mwh(panel).index)
festivo = local["es_festivo"].astype(bool).resample("D").max().reindex(diario.index)
tipos = eda.tipo_dia(diario.index, festivo)
pd.Series(eda.pruebas_calendario(diario * 1000, tipos))

## Temperatura

In [ ]:
resultado = eda.temperatura(serie, local["temperatura_c"].reindex(serie.index))
pd.Series(resultado["spearman_por_hora"]).plot.bar(figsize=(12, 3), title="Spearman demanda-temperatura dentro de cada hora")
{k: v for k, v in resultado.items() if k != "spearman_por_hora"}

## Conclusiones para el modelado

- La serie tiene tendencia: un bosque no extrapola, así que se modelará la
  diferencia semanal o un componente lineal más un árbol sobre el residuo.
- `lag_168` es el rezago más informativo y la base estacional ingenua es la
  referencia obligatoria.
- Festivos y domingos se comportan igual: las banderas del calendario entran.
- El efecto de la temperatura depende de la hora: interacción que captura un
  árbol y no un modelo lineal.